<a href="https://colab.research.google.com/github/machancejoy-max/colab-git-demo-JOY/blob/main/Assignment_6_PAAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import numpy as np
import time
import os
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

# Normalize pixel values (0–255 → 0–1)
x_train = x_train / 255.0
x_test = x_test / 255.0

model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit( x_train, y_train, epochs=15, validation_split=0.2)

test_loss, test_acc = model.evaluate(x_test, y_test)
print("Test Accuracy:", test_acc)
model.save("baseline_cnn.h5")

size_mb = os.path.getsize("baseline_cnn.h5") / (1024 * 1024)
print("Model Size (MB):", size_mb)

sample = x_test[:1]   # one image

start = time.time()
_ = model.predict(sample)
end = time.time()

latency_ms = (end - start) * 1000
print("Inference Latency (ms):", latency_ms)

import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.show()

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.legend()
plt.show()


#Install pruning library
#pip install tensorflow-model-optimization
# Apply Pruning to the Baseline Model

import tensorflow_model_optimization as tfmot

# Define pruning schedule
pruning_params = {
    "pruning_schedule": tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,   # prune 50% of weights
        begin_step=0,
        end_step=1000
    )
}

# Apply pruning wrapper
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

# Compile pruned model
pruned_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

history_pruned = pruned_model.fit(
    x_train, y_train,
    epochs=5,
    validation_split=0.2,
    callbacks=callbacks
)

pruned_model_stripped = tfmot.sparsity.keras.strip_pruning(pruned_model)
pruned_model_stripped.save("pruned_model.h5")

import os

baseline_size = os.path.getsize("baseline_cnn.h5") / (1024*1024)
pruned_size = os.path.getsize("pruned_model.h5") / (1024*1024)

print("Baseline Model Size (MB):", baseline_size)
print("Pruned Model Size (MB):", pruned_size)

import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

quantized_tflite = converter.convert()

with open("quantized_model.tflite", "wb") as f:
    f.write(quantized_tflite)


quant_size = os.path.getsize("quantized_model.tflite") / (1024*1024)
print("Quantized Model Size (MB):", quant_size)

interpreter = tf.lite.Interpreter(model_path="quantized_model.tflite")
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]['index']
output_index = interpreter.get_output_details()[0]['index']

correct = 0
for i in range(1000):  # test on 1000 samples
    img = x_test[i:i+1].astype(np.float32)
    interpreter.set_tensor(input_index, img)
    interpreter.invoke()
    pred = np.argmax(interpreter.get_tensor(output_index))
    if pred == y_test[i]:
        correct += 1

print("Quantized Model Accuracy:", correct / 1000)
#Measure Quantized Inference Latency
start = time.time()
interpreter.invoke()
end = time.time()

print("Quantized Latency (ms):", (end - start) * 1000)

#Quantization may slightly reduce accuracy because weights lose precision.




170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 15s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 42s 32ms/step - accuracy: 0.4347 - loss: 1.5401 - val_accuracy: 0.5139 - val_loss: 1.3339
Epoch 2/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 42s 33ms/step - accuracy: 0.5790 - loss: 1.1811 - val_accuracy: 0.5926 - val_loss: 1.1348
Epoch 3/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 40s 32ms/step - accuracy: 0.6362 - loss: 1.0294 - val_accuracy: 0.6356 - val_loss: 1.0475
Epoch 4/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 40s 31ms/step - accuracy: 0.6724 - loss: 0.9305 - val_accuracy: 0.6632 - val_loss: 0.9546
Epoch 5/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 41s 31ms/step - accuracy: 0.7020 - loss: 0.8502 - val_accuracy: 0.6823 - val_loss: 0.9217
Epoch 6/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 38s 31ms/step - accuracy: 0.7243 - loss: 0.7865 - val_accuracy: 0.6955 - val_loss: 0.8835
Epoch 7/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 38s 30ms/step - accuracy: 0.7426 - loss: 0.7349 - val_accuracy: 0.6926 - val_loss: 0.8887
Epoch 8/15
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 40s 30ms/step - accuracy: 0.7588 -